In [17]:
!pip install POT accelerate

# Analysis of Room Localization and Spatial Embedding System

## Overview
This notebook implements a multi-modal room localization system for the Spot robot using:
- **Vision-Language Models**: SigLIP embeddings combining image and text information
- **Retrieval**: Retrieve using either wasserstein distance or CLIP
- **Spatial Mapping**: A scene graph with room prototypes and node-level embeddings

## Key Components

### Data Structures
- **`room_embeddings`**: (9, D) matrix of room prototype embeddings representing each room's semantic signature
- **`room_labels`**: List of 9 room names (e.g., 'INSITE Lab room 1', 'Miller Street 2', etc.)
- **`room_node_embeddings`**: (N_nodes, D) embeddings for individual nodes/objects tracked in the scene
- **`room_node_labels`**: Integer indices mapping each node to its corresponding room
- **`occupancy_grid`**: Spatial grid encoding occupied/unoccupied regions
- **`room_id_map`**: 2D spatial index mapping grid cells to room IDs

### Localization Pipeline
1. **Detection**: YOLO extracts bounding boxes and class labels from multi-camera feeds
2. **Embedding**: Vision encoder (SigLIP) produces fused embeddings from detected objects
3. **Matching**: Optimal Transport compares the distribution of detections to room-specific node distributions
4. **Prediction**: Returns the room with minimum Wasserstein cost (highest score)

### Visualization
- **`show_frame_grid()`**: Displays synchronized multi-camera views with:
    - Object detections overlaid with confidence scores
    - Predicted room label and confidence
    - Organized camera layout matching robot sensor configuration
    
- **`make_gif()`**: Generates animation sequences showing localization predictions over time

## Results
- 9 distinct rooms with learned semantic embeddings
- 250+ tracked nodes distributed across rooms
- Real-time localization leveraging both visual content and spatial scene graph
- GIF outputs: `spot_multicam_pred_OT_grayscale_nothres.gif` showing full trajectory predictions

In [25]:
from spot.data_loading import SpotDataset
from build_sg_db import *
from Model.models import *
from utils.data_construction.plot_graph import *
import math
from configs.loader import cfg
import pandas as pd
import json
import pickle as pkl
from utils.logger import *
# import ot
from generate_occupancy_field import *

%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [5]:
ds = SpotDataset("dataset/spot/millerst/data")


In [16]:
depth_info = {}

for cam in ds[50].cameras:
    if 'depth' in cam:
        depth_info[cam] = ds[0].cameras[cam]

for k in depth_info:
    print(f"{k}: {np.max(depth_info[k].image)}")

back_depth_in_visual_frame: 376
frontleft_depth_in_visual_frame: 4204
frontright_depth_in_visual_frame: 9262
hand_color_in_hand_depth_frame: 255
hand_depth: 7010
hand_depth_in_hand_color_frame: 7004
left_depth_in_visual_frame: 1736
right_depth_in_visual_frame: 2077


In [24]:
ds[45].odom_T_body.position

array([    -5.6185,      4.2645,     0.22968], dtype=float32)

In [34]:
iphone_odo = pd.read_csv('dataset/3578aa5730/odometry_spot_aligned.csv')
# iphone_odo[iphone_odo[' frame'] == 14230][[' x', ' y', ' z']]
iphone_odo[iphone_odo[' frame'] == 14230][[' x', ' y', ' z']]

,x,y,z
14230,-3.315264,46.636257,0.692944


In [12]:
depth_info[k].image

array([[0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       ...,
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0]], dtype=uint16)

back_depth_in_visual_frame: 131.42537760416667
frontleft_depth_in_visual_frame: 487.79921549479167
frontright_depth_in_visual_frame: 487.251416015625
hand_color_in_hand_depth_frame: 66.8098544973545
hand_depth: 1316.5410923141187
hand_depth_in_hand_color_frame: 1062.7639127604166
left_depth_in_visual_frame: 512.090732421875
right_depth_in_visual_frame: 492.02568684895834


In [19]:
logger = build_logger()

ds = SpotDataset("dataset/spot/millerst/data")
print("----Camera sources available-----")
print("\n".join(ds.camera_sources))
print("\n----Camera Intrinsics-----")
for src, intr in ds.intrinsics.items():
    print(f"  {src}: {intr}")
print("\n----Number of frames-----")
print(len(ds))
sg_path = "graph_dataset/graph.json"
tracker, _ = load_full_tracker("graph_dataset/raw_extration/checkpoints", logger)
assign_rooms_from_json(tracker, "graph_dataset/rooms.json")
# tracker = add_room_att(tracker, sg_path)
graph = json.load(open(sg_path, 'r'))
detection_model = YOLODetector(cfg)
vision_encoder = SiglipModel(cfg)

embeddings_field = np.load("graph_dataset/dense_embedding_field.npy")
occupancy_grid = np.load("graph_dataset/occupancy_grid.npy")
with open("graph_dataset/coarse_embedding_field.pkl", "rb") as f:
    coarse_dict = pkl.load(f)
coarse_embeddings_field = coarse_dict["coarse_field"]
room_id_map = coarse_dict["room_id_map"]
room_id_proto = coarse_dict["room_proto"]
with open("./image_db.pkl", 'rb') as file:
    iphone_image_db = pickle.load(file)

----Camera sources available-----
back_depth_in_visual_frame
back_fisheye_image
frontleft_depth_in_visual_frame
frontleft_fisheye_image
frontright_depth_in_visual_frame
frontright_fisheye_image
hand_color_image
hand_color_in_hand_depth_frame
hand_depth
hand_depth_in_hand_color_frame
left_depth_in_visual_frame
left_fisheye_image
right_depth_in_visual_frame
right_fisheye_image
strayscanner

----Camera Intrinsics-----
  frontleft_depth_in_visual_frame: CameraIntrinsics(fx=256.5978088378906, fy=256.0875549316406, cx=309.02777099609375, cy=230.14013671875, skew_x=0.0, skew_y=0.0, k1=0.0, k2=0.0, k3=0.0, p1=0.0, p2=0.0)
  frontleft_fisheye_image: CameraIntrinsics(fx=256.5978088378906, fy=256.0875549316406, cx=309.02777099609375, cy=230.14013671875, skew_x=0.0, skew_y=0.0, k1=0.0, k2=0.0, k3=0.0, p1=0.0, p2=0.0)
  frontright_depth_in_visual_frame: CameraIntrinsics(fx=257.4794921875, fy=256.9323425292969, cx=320.79443359375, cy=239.1023406982422, skew_x=0.0, skew_y=0.0, k1=0.0, k2=0.0, k3=0.0,

[22:04:55] [INFO] [Resume] Loaded object_0073
[22:04:55] [INFO] [Resume] Loaded object_0074
[22:04:56] [INFO] [Resume] Loaded object_0075
[22:04:56] [INFO] [Resume] Loaded object_0076
[22:04:56] [INFO] [Resume] Loaded object_0077
[22:04:56] [INFO] [Resume] Loaded object_0078
[22:04:56] [INFO] [Resume] Loaded object_0079
[22:04:56] [INFO] [Resume] Loaded object_0080
[22:04:56] [INFO] [Resume] Loaded object_0081
[22:04:56] [INFO] [Resume] Loaded object_0082
[22:04:56] [INFO] [Resume] Loaded object_0083
[22:04:56] [INFO] [Resume] Loaded object_0084
[22:04:56] [INFO] [Resume] Loaded object_0085
[22:04:56] [INFO] [Resume] Loaded object_0086
[22:04:56] [INFO] [Resume] Loaded object_0087
[22:04:56] [INFO] [Resume] Loaded object_0088
[22:04:56] [INFO] [Resume] Loaded object_0089
[22:04:56] [INFO] [Resume] Loaded object_0090
[22:04:56] [INFO] [Resume] Loaded object_0091
[22:04:56] [INFO] [Resume] Loaded object_0092
[22:04:56] [INFO] [Resume] Loaded object_0093
[22:04:56] [INFO] [Resume] Loaded 

In [20]:
print("room_id_map:", room_id_map.shape)
room_labels = []
room_node_embeddings = []
room_node_labels = []
room_embeddings = np.zeros((len(room_id_proto), coarse_embeddings_field.shape[-1]))
for i, (k, v) in enumerate(room_id_proto.items()):
    room_labels.append(k)
    room_embeddings[i] = v
    
for obj in tracker.objects:
    room_node_embeddings.append(obj.clip_ft)
    room_node_labels.append(room_labels.index(obj.room))
    
room_node_labels = np.array(room_node_labels).squeeze()
room_node_embeddings = np.array(room_node_embeddings).squeeze()

# Create ground truth labels based on frame index
def get_ground_truth_label(frame_index):
    """
    Returns the ground truth room label for a given frame index (0-indexed).
    Based on the frame ranges you provided.
    """
    if 0 <= frame_index <= 3:
        return "INSITE Lab room 1"
    elif 4 <= frame_index <= 7:
        return "Miller Street 1"
    elif 8 <= frame_index <= 38:
        return "classroom1"
    elif 39 <= frame_index <= 69:
        return "Miller Street 1"
    elif 70 <= frame_index <= 118:
        return "classroom2"
    elif 119 <= frame_index < len(ds):
        return "Miller Street 2"  # or appropriate final room
    else:
        return "Unknown"

# Create ground truth array for all frames
ground_truth = np.array([get_ground_truth_label(i) for i in range(len(ds))])
gt_numeric = np.array([room_labels.index(get_ground_truth_label(i)) for i in range(len(ds))])

print(f"Ground truth labels created: {len(ground_truth)} frames")
print(f"Unique rooms in ground truth: {np.unique(ground_truth)}")


room_id_map: (681, 1212)
Ground truth labels created: 214 frames
Unique rooms in ground truth: ['INSITE Lab room 1' 'Miller Street 1' 'Miller Street 2' 'classroom1' 'classroom2']


In [21]:
def embed_image_db_batched(vision_encoder, iphone_image_db, batch_size=64):
    """
    Embed all DB images in batches.

    Args:
        vision_encoder: SigLIP / DINO wrapper
        iphone_image_db: dict with key 'image_feats' (list of PIL images)
        batch_size: int

    Returns:
        image_feats: (M, D) float32 numpy array (L2-normalized)
    """

    images = iphone_image_db["image_feats"]
    M = len(images)

    all_feats = []

    for start in range(0, M, batch_size):
        end = min(start + batch_size, M)
        batch = images[start:end]

        feats = vision_encoder.embed_images(batch)  # (B, D)
        feats = np.asarray(feats, dtype=np.float32)

        all_feats.append(feats)

    image_feats = np.concatenate(all_feats, axis=0)

    # ensure L2 normalized (safety)
    image_feats /= (np.linalg.norm(image_feats, axis=1, keepdims=True) + 1e-8)

    return image_feats


iphone_image_db["image_feats_actual"] = embed_image_db_batched(
    vision_encoder,
    iphone_image_db,
    batch_size=64
)

iphone_image_db["image_feats_T"] = iphone_image_db["image_feats_actual"].T

KeyError: 'image_feats'

In [ ]:
iphone_image_db["image_feats_actual"].shape

(9816, 768)

In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics.pairwise import cosine_similarity
import imageio.v2 as imageio

# Try importing POT (Python Optimal Transport)
try:
    import ot
except ImportError:
    print("Warning: 'pot' library not installed. OT localization will fail.")
    ot = None

# --- Configuration ---

CAM_LAYOUT = [
    ["hand_color_image"],
    ["frontright_fisheye_image", "frontleft_fisheye_image"],
    ["left_fisheye_image", "right_fisheye_image"],
    ["back_fisheye_image"],
]

BAD_CHARS = {'$', '#', '@', '!', '%', '^', '&', '*', '(', ')', '-', '+', '=', 
             '{', '}', '[', ']', '|', '\\', ':', ';', '"', "'", '<', '>', ',', '.', '?', '/'}

# --- Localization Logic ---

def get_wasserstein_costs(src_embeddings, room_node_embeddings, room_node_labels, unique_room_labels):
    """
    Computes Wasserstein distance between source detections and specific rooms.
    
    Args:
        src_embeddings: (N, D) embeddings of detected objects.
        room_node_embeddings: (M, D) embeddings of all nodes in the map.
        room_node_labels: (M,) Integer indices mapping nodes to rooms.
        unique_room_labels: List of room names.
        
    Returns:
        costs: (R,) Array of distances to each room.
    """
    if ot is None:
        raise ImportError("Please install 'pot' library to use OT: pip install pot")

    costs = np.zeros((len(unique_room_labels),), dtype=np.float32)
    
    # Uniform weights for source distribution (p)
    n_src = src_embeddings.shape[0]
    p = np.ones((n_src,)) / n_src

    for i in range(len(unique_room_labels)):
        # Get embeddings for nodes belonging to room i
        dst_embeddings = room_node_embeddings[room_node_labels == i, :]
        
        if dst_embeddings.shape[0] == 0:
            costs[i] = np.inf
            continue
            
        # Uniform weights for destination distribution (q)
        n_dst = dst_embeddings.shape[0]
        q = np.ones((n_dst,)) / n_dst
        
        # Cost Matrix: Cosine Distance (1 - Cosine Similarity)
        # M shape: (n_src, n_dst)
        sim_matrix = cosine_similarity(src_embeddings, dst_embeddings)
        M = 1.0 - sim_matrix
        
        # Earth Mover's Distance
        costs[i] = ot.emd2(p, q, M)
        
    return costs

def localize(
    crops, labels, vision_encoder, map_data, lam=0.8, method="OT",
    pooling="mean", eps=1e-8, iphone_image_db=None,
    retrieval_vote="vote",   # "vote" or "max"
    topk_vote=1,             # if >1, allow each crop to vote for top-k
):
    """
    Performs localization based on visual crops and labels.

    Args:
        iphone_image_db: dict with keys:
            - 'image_feats': (M,D) float32
            - 'image_labels': (M,) list[str] or array
        retrieval_vote:
            - "max": choose label of the single most similar DB image over all crops
            - "vote": each crop votes for its best (or top-k) DB label, majority wins
        topk_vote:
            - if retrieval_vote="vote", each crop can vote for its top-k db images.

    Returns:
        prediction_label (str), score (float)
    """
    if len(crops) == 0:
        return "Unknown", 0.0

    room_names = map_data.get('room_labels')  # List[str] for other methods

    # -------------------------
    # 1) Embed detections
    # -------------------------
    # Prefer batching: your encoders (SigLIP/DINO) accept list-of-PIL.
    # If caller passed a single crop per loop previously, fix: pass crops directly.
    try:
        e_img = vision_encoder.embed_images(crops)  # (N,D)
    except Exception:
        # fallback: per-crop (slower but safe)
        e_img = np.stack([np.asarray(vision_encoder.embed_images([c])).squeeze() for c in crops], axis=0)

    e_img = np.asarray(e_img, dtype=np.float32)
    if e_img.ndim == 1:
        e_img = e_img[None, :]

    # # Optional text fusion if encoder supports text
    E = e_img
    # if lam > 0:
    #     try:
    #         # batch embed texts if available
    #         # labels might be list[str] already; if user passes labels[i] as str, we need list
    #         text_list = labels if isinstance(labels, (list, tuple, np.ndarray)) else [labels]
    #         e_txt = vision_encoder.embed_texts(text_list)  # (N,D)
    #         e_txt = np.asarray(e_txt, dtype=np.float32)
    #         if e_txt.ndim == 1:
    #             e_txt = e_txt[None, :]
    #         # fuse
    #         E = lam * e_txt + (1.0 - lam) * e_img
    #     except Exception:
    #         # image-only encoder (e.g., DINO): just use image embeddings
    #         E = e_img

    # L2 normalize query embeddings
    E = E / (np.linalg.norm(E, axis=1, keepdims=True) + eps)  # (N,D)

    # -------------------------
    # 2) Method branches
    # -------------------------
    if method == "CLIP":
        if pooling == "mean":
            q = E.mean(axis=0)
        else:
            q = E.max(axis=0)
        q /= (np.linalg.norm(q) + eps)
        room_feats = map_data['room_embeddings']
        scores = cosine_similarity(q[None, :], room_feats).squeeze()
        best_idx = int(np.argmax(scores))
        return room_names[best_idx], float(scores[best_idx])

    elif method == "OT":
        node_feats = map_data['node_embeddings']
        node_lbls = map_data['node_labels']
        costs = get_wasserstein_costs(E, node_feats, node_lbls, room_names)
        scores = -costs
        best_idx = int(np.argmax(scores))
        return room_names[best_idx], float(scores[best_idx])

    elif method == "image_retrieval":
        if iphone_image_db is None:
            raise ValueError("iphone_image_db is required for method='image_retrieval'")

        db_feats = np.asarray(iphone_image_db["image_feats_actual"], dtype=np.float32)  # (M,D)
        db_labels = np.asarray(iphone_image_db["i_label"])                 # (M,)

        if db_feats.ndim != 2 or E.shape[1] != db_feats.shape[1]:
            raise ValueError(f"Dim mismatch: query {E.shape} vs db {db_feats.shape}")

        # ensure db is normalized (do it once when building db if possible)
        db_feats = db_feats / (np.linalg.norm(db_feats, axis=1, keepdims=True) + eps)

        # Vectorized cosine sim: (N,D) @ (D,M) => (N,M)
        sim = E @ db_feats.T  # (N,M)

        if retrieval_vote == "max":
            # single best match overall (crop, image)
            flat_idx = int(np.argmax(sim))
            i = flat_idx // sim.shape[1]   # crop index
            j = flat_idx % sim.shape[1]    # db image index
            return str(db_labels[j]), float(sim[i, j])

        # vote (default): each crop votes for its top-1 (or top-k) db label
        N, M = sim.shape

        k = int(max(1, topk_vote))
        if k == 1:
            best_j = np.argmax(sim, axis=1)             # (N,)
            best_scores = sim[np.arange(N), best_j]     # (N,)
            voted_labels = db_labels[best_j]            # (N,)
        else:
            # top-k per crop (partial sort)
            topk_j = np.argpartition(-sim, kth=k-1, axis=1)[:, :k]  # (N,k), unordered
            # order within top-k
            topk_s = sim[np.arange(N)[:, None], topk_j]             # (N,k)
            order = np.argsort(-topk_s, axis=1)
            topk_j = topk_j[np.arange(N)[:, None], order]           # (N,k)
            topk_s = topk_s[np.arange(N)[:, None], order]           # (N,k)

            voted_labels = db_labels[topk_j].reshape(-1)            # (N*k,)
            best_scores = topk_s.reshape(-1)                        # (N*k,)

        # Majority vote with tie-break by summed similarity
        # (works with string labels too)
        uniq, inv = np.unique(voted_labels, return_inverse=True)
        counts = np.bincount(inv, minlength=len(uniq))
        sums = np.bincount(inv, weights=best_scores, minlength=len(uniq))

        # pick highest count, break ties by higher summed sim
        # (lexicographic on (count, sum))
        best = np.lexsort((sums, counts))[-1]
        for k in iphone_image_db['label2i']:
            if iphone_image_db['label2i'][k] == uniq[best]:
                pred_label = k

        # score: either avg sim among votes for pred_label, or max sim
        mask = (voted_labels == uniq[best])
        score = float(best_scores[mask].mean()) if mask.any() else float(sums[best] / (counts[best] + eps))
        return pred_label, score

    else:
        raise ValueError(f"Unknown method: {method}")
# --- Visualization Logic ---

def show_frame_grid(
    ds, 
    frame_index, 
    cam_layout=CAM_LAYOUT, 
    detector=None, 
    class_names=None, 
    vision_encoder=None,
    map_data=None,
    localization_method="OT",
    detection_thresh=0.5,
    lam=0.8,
    all_gray=False,
    figsize=(14, 12),
    iphone_image_db=None
):
    sample = ds[frame_index]
    nrows = len(cam_layout)

    # Layout: We add extra space at the bottom or top for the prediction text
    fig, axes = plt.subplots(nrows=nrows, ncols=1, figsize=figsize, constrained_layout=True)
    if nrows == 1: axes = [axes]

    crops = []
    labels = []
    
    # 1. Setup Detector
    if detector is not None and class_names is not None:
        # Filter bad characters from class names
        cls = [n for n in class_names if BAD_CHARS.isdisjoint(n)]
        detector.model.set_classes(cls)
        detector.class_names = cls

    fig.suptitle(f"t={frame_index} | {sample.timestamp}", fontsize=14)

    # 2. Process Images (Plot + Detect + Crop)
    for r, cams in enumerate(cam_layout):
        axes[r].axis("off")
        host = axes[r].inset_axes([0, 0, 1, 1])
        host.axis("off")

        ncols = len(cams)
        for c, cam in enumerate(cams):
            ax = host.inset_axes([c / ncols, 0, 1 / ncols, 1])
            ax.axis("off")

            frame = sample.cameras.get(cam)
            if frame is None or frame.image is None:
                ax.text(0.5, 0.5, "MISSING", ha="center", va="center")
                continue

            if all_gray and "hand" in cam:
                frame.image = np.asarray(Image.fromarray(frame.image).convert('L').convert('RGB'))
            img = frame.image
            ax.imshow(img)
            ax.set_title(cam, fontsize=9)

            if detector is None:
                continue

            # Run Object Detection
            if localization_method!="image_retrieval":
                bbox, cls_idx, scores_det = detector(img)
                
                for i in range(len(bbox)):
                    x1, y1, x2, y2 = bbox[i]
                    
                    if scores_det[i] < detection_thresh:
                        continue

                    # Visualization
                    rect = plt.Rectangle((x1, y1), x2 - x1, y2 - y1, linewidth=2, edgecolor="red", facecolor="none")
                    ax.add_patch(rect)
                    label = detector.class_names[cls_idx[i]]
                    ax.text(x1, max(0, y1 - 10), f"{label}: {scores_det[i]:.2f}", 
                            color="red", fontsize=8, backgroundcolor="white")
                                    
                    # Crop Extraction
                    # Ensure coordinates are within bounds
                    y1_c, y2_c = max(0, math.floor(y1)), min(img.shape[0], math.ceil(y2))
                    x1_c, x2_c = max(0, math.floor(x1)), min(img.shape[1], math.ceil(x2))
                    
                    crop = img[y1_c:y2_c, x1_c:x2_c, :]
                    
                    if crop.size > 0:
                        crops.append(crop)
                        labels.append(label)
            else:
                crops.append(img)
                labels.append("NA")

    # 3. Run Localization (Prediction)
    pred_text = "Localization: N/A"
    if vision_encoder is not None:
        pred_room, pred_score = localize(
            crops, labels, vision_encoder, map_data, lam=lam, method=localization_method, iphone_image_db=iphone_image_db
        )
        pred_text = f"Pred: {pred_room}\nScore: {pred_score:.4f}"
    elif len(crops) == 0 and detector is not None:
        pred_text = "Pred: Uncertain (No Detections)"

    # 4. Add Prediction Overlay
    # We place a text box in the top-right corner of the figure
    fig.text(
        0.6, 0.6, 
        pred_text, 
        fontsize=12, 
        color='white', 
        ha='right', 
        va='top',
        bbox=dict(boxstyle="round,pad=0.5", fc="black", ec="none", alpha=0.7)
    )
    fig.text(
        0.45, 0.55, 
        f"GT: {ground_truth[frame_index]}", 
        fontsize=12, 
        color='green' if ground_truth[frame_index] == pred_room else 'red', 
        ha='left', 
        va='top',
        bbox=dict(boxstyle="round,pad=0.5", fc="black", ec="none", alpha=0.7)
    )

    return fig, ground_truth[frame_index] == pred_room

# --- Main Driver ---

def make_gif(
    ds,
    out_path="spot_multicam.gif",
    cam_layout=CAM_LAYOUT,
    start=0,
    end=None,
    step=1,
    detector=None,
    class_names=None,
    vision_encoder=None,
    map_data=None, # Contains embeddings and labels
    detection_thresh=0.5,
    localization_method="OT",
    all_gray=False,
    lam=0.8,
    fps=6,
    figsize=(14, 12),
    iphone_image_db=None
):
    """
    Generates a GIF by iterating through the dataset and visualizing predictions.
    """
    if end is None:
        end = len(ds)

    frames = []

    print(f"Generating GIF from tnepotism={start} to {end}...")
    accuracy = 0
    for t in range(start, end, step):
        # All logic is now encapsulated in show_frame_grid
        fig, is_correct = show_frame_grid(
            ds=ds, 
            frame_index=t, 
            cam_layout=cam_layout, 
            detector=detector, 
            class_names=class_names, 
            vision_encoder=vision_encoder,
            map_data=map_data,
            all_gray=all_gray,
            lam=lam,
            detection_thresh=detection_thresh,
            localization_method=localization_method,
            figsize=figsize,
            iphone_image_db=iphone_image_db
        )

        # Render figure to numpy array
        fig.canvas.draw()
        rgba = np.asarray(fig.canvas.buffer_rgba())
        frames.append(rgba[..., :3].copy()) # Drop Alpha channel
        plt.close(fig)
        accuracy += int(is_correct)
        
    imageio.mimsave(out_path, frames, fps=fps)
    print(f"Method: {localization_method}")
    print("Parameters: ")
    print(f"- Detection Threshold: {detection_thresh}")
    print(f"- lambda (Image/Text weight): {lam}")
    print(f"Localization Accuracy: {accuracy}/{len(frames)} = {accuracy/len(frames)*100:.2f}%")
    print(f"[saved] {out_path}")

In [ ]:
dino = DinoModel(cfg)

In [ ]:
iphone_image_db["image_feats_actual"] = embed_image_db_batched(
    vision_encoder,
    iphone_image_db,
    batch_size=64
)

iphone_image_db["image_feats_T"] = iphone_image_db["image_feats_actual"].T

In [ ]:
map_data = {
    'room_embeddings': room_embeddings,
    'room_labels': room_labels,
    'node_embeddings': room_node_embeddings,
    'node_labels': room_node_labels
}

fig = show_frame_grid(
    ds, 
    80, 
    cam_layout=CAM_LAYOUT, 
    detector=detection_model, 
    class_names=[n['class_name'] for n in graph['nodes']], 
    vision_encoder=dino,
    map_data=map_data,
    lam=0,
    detection_thresh=0,
    all_gray=True,
    localization_method="image_retrieval",
    figsize=(14, 12),
    iphone_image_db=iphone_image_db
)

['white desktop printer', 'red office chair', 'Large multi-paned window', 'wooden office shelf', 'white office printer', 'beige air purifier', 'wooden display board', 'wooden door handle', 'Whiteboard with writing', 'Red office chair', 'red office stool', 'red office chair', 'Black flat screen', 'blue office chair', 'academic research poster', 'White rectangular board', 'blue office chair', 'Large white whiteboard', 'information poster images', 'Poster on wall.', 'wall posters display', 'research poster board', 'red office chair', 'wooden door frame', 'White trash can', 'White door frame', 'Gray metal panel', 'wooden office door', 'empty classroom tables', 'empty classroom desks', 'blue trash can', 'Gray plastic trash', 'Large white board', 'Large white board.', 'Large white board', 'Round white table.', 'Large white board', 'Black electronic rack.', 'white office table', 'Black flat screen', 'White table surface', 'white rectangular surface', 'Black flat screen', 'white rolling table'

RuntimeError: No active exception to reraise

In [ ]:
graph.keys()

dict_keys(['nodes', 'edges'])

In [ ]:
make_gif(
    ds,
    out_path="spot_multicam_pred_imgretriev_siglip_.gif",
    cam_layout=CAM_LAYOUT,
    start=0,
    end=len(ds),
    step=1,
    detector=detection_model,
    class_names=[n['class_name'] for n in graph['nodes']],
    vision_encoder=vision_encoder,
    map_data=map_data, # Contains embeddings and labels
    localization_method="image_retrieval",
    detection_thresh=0.0,
    lam = 0,
    all_gray=False,
    fps=6,
    figsize=(14, 12),
    iphone_image_db=iphone_image_db
)

Generating GIF from t=0 to 214...
Method: image_retrieval
Parameters: 
- Detection Threshold: 0.0
- lambda (Image/Text weight): 0
Localization Accuracy: 90/214 = 42.06%
[saved] spot_multicam_pred_imgretriev_siglip_.gif


![GIF 1](spot_multicam_pred_OT_grayscale_nothres.gif)


In [ ]:
dino_model = DinoModel(cfg)

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


[DINO] Loading facebook/dinov2-large → cuda


model.safetensors:   0%|          | 0.00/1.22G [00:00<?, ?B/s]

In [ ]:
test_im = iphone_image_db["images"][:2]

# move rgb channel to start
test_im = np.transpose(test_im, (0, 3, 1, 2))
test_im.shape

(2, 3, 960, 720)

In [ ]:
out = dino_model.embed_images_by_patch(test_im)

In [ ]:
out.shape

(2, 16, 16, 1024)